# Act 1 — Inside one VPC

Before you can connect to anything else, you need to lay out the network your resources sit inside. That is the VPC — your own isolated piece of AWS networking.

Inside one VPC you make a fixed set of decisions. What address range to use, how to split it into subnets across Availability Zones, what routes those subnets follow, what traffic is allowed in, what traffic is allowed out, how a private instance reaches the internet, and how you get a shell on something that has no public IP. This act walks those decisions in the order you actually make them when standing up a fresh VPC.

## What a VPC is

A **VPC** — Virtual Private Cloud — is your own logically isolated network inside AWS. Every resource that touches the network lives in one: EC2 instances, RDS databases, Lambda functions attached to a VPC, ECS tasks, load balancers. The VPC defines an IP address space, how that space is sliced into subnets, what traffic can leave or enter, and how to reach the rest of the world — other VPCs, AWS services, on-premises data centres, the public internet.

## VPC and CIDR

A VPC is **region-scoped** — it spans every Availability Zone in a region. Subnets inside the VPC are AZ-scoped. You pick an IPv4 CIDR block when you create it, anywhere from `/16` (65,536 addresses) down to `/28` (16 addresses), and you can add **secondary CIDR blocks** later if you outgrow the original allocation. IPv6 is optional — AWS hands out a `/56` and subnets carve out `/64` chunks from it.

A few practical notes on planning. CIDRs cannot overlap with anything you intend to peer with, so pick ranges that leave room across VPCs (`10.0.0.0/16`, `10.1.0.0/16`, `10.2.0.0/16` for distinct environments, not the same `10.0.0.0/16` three times). The RFC 1918 private ranges — `10.0.0.0/8`, `172.16.0.0/12`, `192.168.0.0/16` — are the conventional space, but any IPv4 range works; AWS will route it.

Every account also gets a **default VPC** per region with `172.31.0.0/16`, one public subnet per AZ, and an Internet Gateway already attached. It's fine for quick experiments and disastrous for production — production wants an intentional public/private subnet design, which is what we'll build next.

**Tenancy** is set at the VPC level: default tenancy shares physical hardware with other customers (the normal case); dedicated tenancy puts every instance in the VPC on dedicated hardware at significantly higher cost. It's for regulatory edge cases, not a default choice.

## Subnets

A subnet is a slice of the VPC CIDR confined to a single Availability Zone. The labels **public** and **private** are not properties of the subnet — they're a consequence of the route table the subnet is associated with. A subnet is "public" because its route table has a route to an Internet Gateway; "private" because it doesn't. The same subnet flips identity by re-associating it with a different route table.

AWS reserves **five IP addresses** in every subnet — the first four and the last one. For `10.0.1.0/24`:

| Address | Reserved for |
|---|---|
| `10.0.1.0` | Network address |
| `10.0.1.1` | VPC router |
| `10.0.1.2` | DNS server (base + 2) |
| `10.0.1.3` | Future use |
| `10.0.1.255` | Broadcast (not used, but reserved) |

So a `/24` gives 251 usable IPs, not 256. A `/27` — 32 total — gives 27. This matters when you size small subnets for a Lambda VPC attachment or an interface endpoint and find yourself short.

Each subnet has an **auto-assign public IP** flag — on for default-VPC public subnets, off in custom VPCs. Turn it on for subnets where you want instances to be internet-reachable on launch, off everywhere else.

The canonical multi-AZ pattern is three tiers across two or three AZs — public subnets for load balancers and NAT, private subnets for application servers, and isolated private subnets for databases. The point isn't the count; it's that AZ failure should never take more than one tier offline.

## Internet Gateway and Route Tables

An **Internet Gateway (IGW)** is a horizontally-scaled, redundant component that lets VPC traffic reach the internet and back. One per VPC, attached to the VPC, then referenced from route tables. It also performs the NAT step that maps an instance's private IP to its public or Elastic IP on outbound traffic.

For an instance to be reachable from the internet, it needs **three things together**: a public or Elastic IP, a route to the IGW in its subnet's route table, and a security group rule allowing the inbound port. Miss any one and the traffic is dropped — often invisibly.

A **route table** is a list of destination CIDR to target mappings. Every subnet is associated with exactly one route table (the default one if you don't pick). The simplest public/private split looks like this:

| Route table | Destination | Target |
|---|---|---|
| Public | `10.0.0.0/16` | local |
| Public | `0.0.0.0/0` | igw-xxx |
| Private | `10.0.0.0/16` | local |
| Private | `0.0.0.0/0` | nat-xxx |

The **local** route covers the VPC CIDR and is implicit — you cannot delete it. Routing uses **longest prefix match**: the most specific matching route wins. A `0.0.0.0/0` default route is the last-resort target when no more specific route matches.

## NAT Gateway

Private subnets often need outbound internet access — to call package mirrors, hit AWS service endpoints that aren't reached privately, or pull software updates — but they should not be reachable from the internet inbound. A **NAT Gateway** is the standard answer: a managed component that sits in a public subnet, holds an Elastic IP, and translates outbound traffic from private subnets so it appears to come from that Elastic IP.

| Property | Detail |
|---|---|
| Placement | Public subnet, in a specific AZ |
| Reachability | Stateful — outbound only from private; inbound responses allowed automatically |
| Bandwidth | Auto-scales up to 100 Gbps |
| Cost | Hourly + data-processing per GB |
| Security groups | None — NAT Gateway cannot have an SG attached |

**The HA caveat** is important: a NAT Gateway lives in one AZ. If you put a single NAT Gateway in AZ-a and that AZ fails, every private subnet routed through it loses internet — even private subnets in other AZs. The pattern is one NAT Gateway per AZ, with each AZ's private route table pointing at its own AZ's NAT Gateway. It costs more, but a regional VPC built around a single-AZ NAT is not really regional.

The legacy **NAT Instance** — a self-managed EC2 doing the same job — still exists, but the modern default is always NAT Gateway. NAT Instance is only kept around for cost-sensitive cases or when you specifically need to attach a security group at the NAT layer.

## Security Groups

A security group is a virtual firewall attached to an **ENI** — the network interface on an instance, load balancer, RDS database, Lambda function, anything with an IP in your VPC. The model has four properties that determine almost everything about how you design network access.

- **Stateful** — if you allow inbound on port 443, the return traffic is allowed automatically. You don't write a matching outbound rule.
- **Allow rules only** — there is no "deny" in a security group. You permit what you want and everything else is denied implicitly.
- **Multiple SGs per ENI** — rules combine additively; if any SG allows the traffic, it's allowed.
- **SG references** — a rule can reference another security group as its source instead of an IP range. "Allow port 8080 from the ALB SG" is more expressive and more maintainable than "allow from 10.0.1.0/24" because it tracks instances automatically as they scale.

The canonical pattern is the layered chain: the ALB security group allows port 443 from the internet; the app security group allows its port from the ALB SG; the database security group allows its port from the app SG. Each layer only trusts the layer above it, and the references survive any number of instance launches and terminations.

The **default security group** in a fresh VPC has an inbound rule that allows all traffic from itself — meaning every instance in the default SG can reach every other instance in the default SG. It's convenient and a common source of accidental open access; new workloads should get a fresh, narrowly-scoped SG.

## Network ACLs

Network ACLs sit one layer up — at the **subnet** boundary — and evaluate traffic entering or leaving the subnet, separately from any security group. They look superficially similar to security groups but have inverted properties:

| | Security Group | NACL |
|---|---|---|
| Level | ENI (instance) | Subnet |
| State | Stateful | **Stateless** |
| Rules | Allow only | **Allow and Deny** |
| Evaluation | All rules combined | **Ordered**, first match wins |
| Default (custom) | Deny all inbound | Deny all (custom NACL) |
| Default (the auto-created "default" NACL) | — | **Allow all in and out** |

The stateless property is the trap. Because return traffic is not automatically allowed, you have to permit both directions explicitly — and the response to an inbound request comes back on an **ephemeral port** (Linux uses 1024–65535). If you allow inbound on 443 but forget to allow outbound on the ephemeral range, the connection completes the handshake and then hangs. This is the most common NACL misconfiguration.

In practice you let the default NACL stay wide open and do all the firewalling with security groups, then reach for custom NACLs only for the specific cases where security groups can't help: blocking a specific malicious IP at the subnet edge, enforcing a deny-list, or putting a hard boundary between environments. SGs are the workhorse; NACLs are the bouncer.

## Elastic IPs

An **Elastic IP** is a static IPv4 address you own at the account level and can attach to any ENI in your account. Two properties make it useful: it survives instance stops (a regular auto-assigned public IP is released on stop), and it can be remapped from one instance to another in minutes — the textbook failover pattern.

Elastic IPs are free while attached to a running instance, billed at a small hourly rate when allocated but not in use (to discourage hoarding the limited IPv4 space), and capped at 5 per region by default — raise via support. The architectural rule is: use them where external clients need a fixed IP (NAT Gateway, VPN endpoint, on-prem firewall allowlist). Don't put them on application servers — that's what load balancers are for.

## DNS Inside the VPC

Every VPC has two DNS toggles. **enableDnsSupport** turns on the AWS-managed DNS resolver at the VPC base CIDR plus 2, which resolves AWS service endpoints and Route 53 private hosted zones. **enableDnsHostnames** auto-assigns public DNS names to instances with public IPs. Both default on in the default VPC and off in custom VPCs; flip both on if you intend to use Route 53 private hosted zones to resolve internal names.

The VPC resolver also forwards to **Route 53 Resolver** endpoints — outbound endpoints for forwarding queries to on-premises DNS servers, inbound endpoints for the reverse. That's how a hybrid network resolves names across the boundary in both directions.

## VPC Flow Logs

Flow Logs capture the **metadata** of every IP packet flowing through a VPC, subnet, or specific ENI — source and destination IP, port, protocol, byte count, packet count, accept-or-reject status — but not packet contents. Destinations are S3 for long-term retention and Athena querying, CloudWatch Logs for real-time alerting, or Kinesis Data Firehose to stream into a SIEM.

Flow Logs are the tool of last resort when traffic isn't doing what you expect. A security group or NACL silently dropping packets shows up as REJECT entries on the offending tuple, and that's usually enough to identify which rule needs to change. A few things they don't capture — instance metadata service traffic to 169.254.169.254, DHCP, DNS queries to the VPC resolver, and traffic to the VPC router — but for everything else they're the ground truth.

## Reaching Private Instances

The historical pattern for getting a shell on a private instance was a **bastion host** — a hardened EC2 in a public subnet with SSH locked down to your office IP, used as a jump box into the private network. It still works, and SSH agent forwarding keeps your private key off the bastion itself.

The modern answer is to skip the bastion entirely. **AWS Systems Manager Session Manager** opens a shell to any instance running the SSM agent — no open SSH port, no public subnet, no key pair, every session audited in CloudTrail and optionally recorded to S3. **EC2 Instance Connect** is similar for SSH-style access, pushing a one-time key via the AWS API. Both work because the instance initiates the connection out to AWS (or, with a VPC endpoint for SSM, even that stays private), so nothing inbound needs to be exposed. New infrastructure should default to Session Manager; bastions are now a fallback, not the starting point.

In [ ]:
import boto3

ec2 = boto3.client("ec2", region_name="us-east-1")

# VPC + DNS hostnames on for private hosted zones to work later
vpc = ec2.create_vpc(CidrBlock="10.0.0.0/16")["Vpc"]
ec2.modify_vpc_attribute(VpcId=vpc["VpcId"], EnableDnsHostnames={"Value": True})

# Subnets across two AZs — two public, two private
subnets = {}
for cidr, az, name, public in [
    ("10.0.1.0/24", "us-east-1a", "public-1a",  True),
    ("10.0.2.0/24", "us-east-1b", "public-1b",  True),
    ("10.0.3.0/24", "us-east-1a", "private-1a", False),
    ("10.0.4.0/24", "us-east-1b", "private-1b", False),
]:
    s = ec2.create_subnet(VpcId=vpc["VpcId"], CidrBlock=cidr, AvailabilityZone=az)["Subnet"]
    subnets[name] = s["SubnetId"]
    if public:
        ec2.modify_subnet_attribute(SubnetId=s["SubnetId"], MapPublicIpOnLaunch={"Value": True})

# IGW + public route table — 0.0.0.0/0 to the IGW
igw = ec2.create_internet_gateway()["InternetGateway"]
ec2.attach_internet_gateway(InternetGatewayId=igw["InternetGatewayId"], VpcId=vpc["VpcId"])
pub_rt = ec2.create_route_table(VpcId=vpc["VpcId"])["RouteTable"]
ec2.create_route(RouteTableId=pub_rt["RouteTableId"],
                 DestinationCidrBlock="0.0.0.0/0",
                 GatewayId=igw["InternetGatewayId"])
for name in ["public-1a", "public-1b"]:
    ec2.associate_route_table(RouteTableId=pub_rt["RouteTableId"], SubnetId=subnets[name])

# One NAT Gateway per AZ — each private subnet routes to its AZ's NAT
for pub, priv in [("public-1a", "private-1a"), ("public-1b", "private-1b")]:
    eip = ec2.allocate_address(Domain="vpc")
    nat = ec2.create_nat_gateway(SubnetId=subnets[pub],
                                 AllocationId=eip["AllocationId"])["NatGateway"]
    rt = ec2.create_route_table(VpcId=vpc["VpcId"])["RouteTable"]
    ec2.create_route(RouteTableId=rt["RouteTableId"],
                     DestinationCidrBlock="0.0.0.0/0",
                     NatGatewayId=nat["NatGatewayId"])
    ec2.associate_route_table(RouteTableId=rt["RouteTableId"], SubnetId=subnets[priv])

# Act 2 — Connecting VPCs and AWS services privately

One VPC is rarely enough. Real organisations end up with many — separated by account, by environment, by team, by region — and those VPCs need to talk to each other. The services your code calls — S3, DynamoDB, SQS, Lambda — also live somewhere in AWS, and you usually want to reach them without routing through the public internet at all.

This act covers four answers to those two problems. **VPC peering** for small, point-to-point connections between two VPCs. **VPC endpoints** for reaching AWS services privately. **PrivateLink** for exposing your own service to other VPCs without peering. And **Transit Gateway** for the case where the number of VPCs has grown past what a peering mesh can hold.

## VPC Peering

VPC Peering is a direct, point-to-point connection between two VPCs that lets instances in each reach the other by private IP, as if they were on the same network. It works **same account or cross-account, same region or cross-region**, using AWS's backbone with no bandwidth cap.

The constraints are sharper than people expect. CIDR blocks **cannot overlap** — if both VPCs use `10.0.0.0/16`, peering simply won't establish. Peering is **non-transitive**: if A peers with B and B peers with C, traffic from A still cannot reach C through B; you need a direct A-to-C peering. After accepting the request, you have to **update route tables on both sides** to point at the peering connection — without the routes, the connection exists but no traffic flows.

Security groups in a peered VPC can be referenced in another VPC's SG rules within the same region — "allow port 5432 from the app-tier SG in the other VPC" works directly, which is one of the nicer properties of peering.

The right time for peering is a small number of VPCs — two, three, maybe four — that need to talk. The number of peering connections grows as N times (N minus 1) over 2, so the full mesh becomes unmanageable fast. At that point you switch to Transit Gateway.

## VPC Endpoints

A VPC Endpoint lets a VPC reach AWS services **privately**, without routing through an Internet Gateway, a NAT Gateway, or a VPN. It matters for two reasons: NAT bandwidth and per-GB charges add up at scale, and many regulated workloads can't have traffic to AWS leave the AWS network at all.

There are two endpoint types, and the distinction is the whole story.

| | Gateway Endpoint | Interface Endpoint |
|---|---|---|
| Services | **S3 and DynamoDB only** | Most AWS services |
| Mechanism | Adds a route in route tables to a prefix list | Creates an ENI with a private IP in your subnet |
| Cost | **Free** | Per hour + per GB processed |
| DNS | Public DNS resolves to the endpoint via route | Optional **private DNS** — service hostname resolves to the ENI's private IP |
| Scope | Region | Per subnet (one ENI per AZ) |

A **Gateway Endpoint** for S3 is the simple architectural win: any private subnet that calls S3 gets it directly across the AWS backbone with no NAT data charges. You attach the endpoint to specific route tables and AWS adds a prefix-list route. It's free, so there's no reason not to use it.

**Interface Endpoints** are powered by PrivateLink — an ENI lives in your subnet with a private IP, and traffic destined for the service goes there. They support **endpoint policies** that restrict which principals and which actions can use the endpoint, which is a useful belt-and-braces guard alongside the service's own IAM policies. Private DNS, when enabled, makes this transparent to your code — `sqs.us-east-1.amazonaws.com` resolves to the private ENI IP automatically, no SDK changes.

## PrivateLink for Your Own Services

The same PrivateLink machinery that powers Interface Endpoints can be turned around to expose **your own service** in your VPC to consumers in other VPCs — without peering, without exposing the service to the internet, and without worrying about CIDR overlap.

The provider side puts a **Network Load Balancer** in front of the service and creates a **VPC Endpoint Service** that registers the NLB. The consumer side creates an Interface Endpoint that targets that service. Traffic flows over the AWS backbone from the consumer's private IP to the provider's NLB, completely privately. The provider can permit specific AWS accounts and even specific principals to discover and connect to the service.

This is the right answer for two shapes of problem. **SaaS vendors on AWS** use PrivateLink to give thousands of customer VPCs a private connection to their service without any of them peering. **Internal microservices shared across accounts** use it to expose, say, a shared identity service across an Organization without setting up a peering mesh. The shorthand: peering connects whole VPCs, PrivateLink exposes a single service. PrivateLink scales to thousands of consumers; peering does not.

## Transit Gateway

When the number of VPCs that need to talk grows past a handful, peering's full mesh stops being sustainable. **Transit Gateway (TGW)** replaces the mesh with a **hub-and-spoke** topology — every VPC attaches once to the TGW, and the TGW routes between them. Ten VPCs that would have needed 45 peering connections need 10 attachments instead.

TGW is regional but supports rich attachment types: VPCs, Site-to-Site VPN connections, Direct Connect Gateways, and **other Transit Gateways** for inter-region peering. Routing is **transitive** by default — any attachment can reach any other — which immediately raises the question of how you keep environments separated.

The answer is **TGW route tables**. A TGW has multiple route tables, and each attachment is associated with one of them. A dev attachment associated with the dev route table sees other dev VPCs but not prod; the prod route table sees prod VPCs and the on-prem attachment but not dev. This is how you build a single global network with multiple isolation domains.

Transit Gateway can be **shared across AWS accounts via Resource Access Manager** — one networking-hub account owns the TGW, and every other account in the Organization attaches its VPCs to it. This is the default architecture for any non-trivial multi-account network on AWS.

The rule of thumb: peering for two to four VPCs in the same region with simple, stable needs; Transit Gateway for any meaningful scale, any multi-region pattern, or any need to mix VPN and Direct Connect attachments into the same routing fabric.

In [ ]:
import boto3, json

ec2 = boto3.client("ec2", region_name="us-east-1")

# S3 Gateway Endpoint — free, scoped by endpoint policy to one bucket
ec2.create_vpc_endpoint(
    VpcId="vpc-0aaa111",
    ServiceName="com.amazonaws.us-east-1.s3",
    VpcEndpointType="Gateway",
    RouteTableIds=["rtb-0private1", "rtb-0private2"],
    PolicyDocument=json.dumps({
        "Statement": [{
            "Effect": "Allow", "Principal": "*",
            "Action": ["s3:GetObject", "s3:PutObject"],
            "Resource": "arn:aws:s3:::my-app-bucket/*",
        }],
    }),
)

# SSM Interface Endpoint — lets private instances use Session Manager without NAT
ec2.create_vpc_endpoint(
    VpcId="vpc-0aaa111",
    ServiceName="com.amazonaws.us-east-1.ssm",
    VpcEndpointType="Interface",
    SubnetIds=["subnet-0priv1a", "subnet-0priv1b"],
    SecurityGroupIds=["sg-0endpoint"],
    PrivateDnsEnabled=True,   # ssm.us-east-1.amazonaws.com resolves to the ENI
)
# Note: full SSM coverage also needs the ssmmessages and ec2messages endpoints

In [ ]:
import boto3

ec2 = boto3.client("ec2", region_name="us-east-1")

# Transit Gateway — central hub for many VPCs
tgw = ec2.create_transit_gateway(
    Description="central hub",
    Options={
        "DefaultRouteTableAssociation": "enable",
        "DefaultRouteTablePropagation": "enable",
        "AutoAcceptSharedAttachments": "disable",
        "DnsSupport": "enable",
    },
)["TransitGateway"]

# Attach a VPC — one subnet per AZ for the TGW ENI
ec2.create_transit_gateway_vpc_attachment(
    TransitGatewayId=tgw["TransitGatewayId"],
    VpcId="vpc-0aaa111",
    SubnetIds=["subnet-0priv1a", "subnet-0priv1b"],
)
# Share via RAM to attach VPCs owned by other accounts in the org

# Act 3 — Reaching on-premises

The third connectivity problem is the one outside AWS entirely — your data centre, your colocation facility, your office network. There are two ways to bridge that gap.

A **Site-to-Site VPN** runs an encrypted tunnel over the public internet. Quick to stand up, encrypted by default, limited in throughput and latency variance by the fact that it shares the public internet with everyone else.

**Direct Connect** is a dedicated physical fibre from your facility to an AWS edge location. It bypasses the public internet entirely, gives you predictable bandwidth and latency, and takes weeks to provision. Production hybrid networks almost always use both — DX as the primary path and VPN as the same-day fallback.

## Site-to-Site VPN

A Site-to-Site VPN runs an **IPsec encrypted tunnel** between your on-premises network and AWS, over the public internet. It's the fastest way to connect a data centre to a VPC — set up in minutes, no physical install, and encrypted by default.

The three components: a **Virtual Private Gateway (VGW)** attached to your VPC on the AWS side (or a TGW with a VPN attachment for the multi-VPC case); a **Customer Gateway (CGW)** representing your on-prem VPN device's public IP; and the **VPN Connection** that runs the IPsec tunnels between them. Every VPN connection has **two tunnels** to two AWS-side endpoints by design — configure both on your on-prem device and the second is automatic failover when the first goes down.

Routing can be **static** (you tell each side which CIDRs are on the other) or **dynamic via BGP** (the two sides exchange routes automatically). BGP is the default for any non-trivial topology and is required for **VPN CloudHub** — the pattern where multiple branch offices connect to a single VGW and reach each other through AWS as the hub.

The limits are honest: each tunnel caps at about 1.25 Gbps, and the path runs over the public internet, so latency varies. For low bandwidth, quick rollout, or backup-path scenarios, Site-to-Site VPN is the right tool. For sustained high throughput or strict latency requirements, you move to Direct Connect.

## Direct Connect

**Direct Connect (DX)** is a dedicated physical fibre link from your data centre (or colocation facility) to an AWS Direct Connect location — bypassing the public internet entirely. It gives you consistent bandwidth, predictable latency, and a private path for compliance regimes that prohibit data transit over the internet.

A single DX connection carries one or more **Virtual Interfaces (VIFs)**, each targeting a different kind of resource:

| VIF type | Reaches | Use |
|---|---|---|
| **Private VIF** | VPC resources by private IP | EC2, RDS, ELB in a VPC |
| **Public VIF** | AWS public service endpoints | S3, DynamoDB, public APIs — over the dedicated link |
| **Transit VIF** | A Transit Gateway via a DX Gateway | Many VPCs through one DX |

A **Direct Connect Gateway** lets one DX connection reach VPCs in **multiple regions** — without it, you'd need a separate DX into each region, which is impractical. Pair the DX Gateway with a Transit Gateway via a Transit VIF and you have a single physical connection feeding an entire global network of VPCs.

Two properties trip people up. DX is **not encrypted by default** — it's a private physical link, but if you need cryptographic encryption (HIPAA, certain financial regulations) you run an IPsec VPN tunnel over the DX connection on top. And **provisioning takes weeks to months** — cross-connects have to be physically installed at a partner facility — so you cannot decide to add Direct Connect on the day you need it. The standard hybrid pattern is therefore DX as the primary path with a Site-to-Site VPN as same-day failover and backup, so you have something resilient before the DX even goes live.

For true production-grade resiliency the recommendation is **two DX connections at two different DX locations** terminating in two different VGWs or TGW attachments — single-location DX has been knocked out by facility events, and a single connection at one location is a single point of failure regardless of how reliable AWS makes the rest of the path.

# Act 4 — Picking the right connectivity option

You now have every connectivity tool on the table. The remaining question is matching the tool to the problem. Once you frame the question correctly — how many VPCs, private or internet, AWS service or on-prem, how much bandwidth, how soon — the answer is usually one specific service. The decision table below consolidates the choices.

## Picking a Connectivity Option

The connectivity layer has the most options in AWS networking — and the question is rarely "is this possible" but "which one fits". A compact decision table:

| Need | Reach for |
|---|---|
| Connect two to four VPCs, same or different region | **VPC Peering** |
| Connect many VPCs with transitive routing and central control | **Transit Gateway** |
| Reach S3 or DynamoDB privately from a VPC | **Gateway Endpoint** (free) |
| Reach other AWS services privately from a VPC | **Interface Endpoint** (PrivateLink) |
| Expose your own service in your VPC to many consumer VPCs | **PrivateLink** — NLB + endpoint service |
| Quick, encrypted, low-bandwidth on-prem to AWS | **Site-to-Site VPN** |
| High bandwidth, low jitter, no internet path | **Direct Connect** |
| Multiple on-prem sites talking to each other via AWS | **VPN CloudHub** (with BGP) |
| Direct Connect to VPCs in multiple regions | **Direct Connect Gateway** + Transit VIF |
| DX that must be cryptographically encrypted | Run a **VPN tunnel over the DX** |
| Production-grade hybrid resiliency | **DX as primary + VPN as failover**, ideally **two DX locations** |

The service that solves the problem is usually obvious once you've framed it in those terms — "how many VPCs", "private to AWS", "to on-prem", "how fast", "how soon". Networking on AWS is mostly about asking those questions in the right order.